
# <font color="green">Function calls (exp(x) + exp(-x))</font>

## Problem

* Write a function `expsum` that takes a `double` $x$ and computes
$$ \mbox{expsum}(x) = \exp(x) + \exp(-x) $$
* Note that you need to call `exp` twice. After the first call `exp(x)` returns, the argument `x` is no longer in `d0`, so you must arrange to still have `x` (or `-x`) available to compute the second call. Save what you need across the first call.
* That is, translate the following C function into assembly:
```
double expsum(double x) {
  return exp(x) + exp(-x);
}
```
* Fill in the skeleton `expsum.s` (after `// ------- write your answer here -------`). You will need to call `exp` from the math library.
* The checker `check_expsum.c` verifies your result (it is linked with `-lm`). If you see `OK`s and no errors, you are done.

## Hints

* If a function calls another function, its assembly becomes more complex, because:
  * calling a function with `bl` overwrites `x30` (the link register), so `x30` must be preserved on the stack;
  * that means the stack (`sp`) must be extended and the frame pointer (`x29`) set, so `x29` must be preserved too.
* In summary, a function that makes a call typically does something like
```
        stp     x29, x30, [sp, -16]!
```
to extend the stack and preserve `x29` and `x30` before the call, and restores them before returning.
* A key subtlety: a call may overwrite registers (including the argument register `d0`). So if you still need a value *after* the call, you must save it somewhere that survives the call --- typically on the stack, or in a callee-saved register. This problem is designed to exercise exactly that.
* The *Observe* cells below contain `sigmoid.c`. Compile it and look at how the stack frame is set up and how `exp` is called with `bl`.
* For details, study how a function call works in the [How Programming Languages Work (Basics)](https://taura.github.io/programming-languages/slides/05-implementation-basics.pdf) slide deck.



# 1. AI Tutor
## 1-1. Prepare
* Your personal AI tutor is provided for questions and feedback.
* Execute the following cell before you use it.

In [ ]:
import heytutor

## 1-2. Examples
* A general question
```
%%hey
What does the `ldr` instruction do in ARM64?
```

* A hint on this specific problem
```
%%hey problem_file=expsum.md
Give me a hint on this problem.

{problem}
```

* Builtin variables usable in `%%hey` cells
  * `{file:FILENAME}` is the content of FILE
  * `{bash[-1]}` is the output of the last `%%bash_` cell, `{bash[-2]}` the second last, etc.
  * `{problem}` is the content of the file you specify by `%%hey problem_file=foo.md`
  * `{answer}` is the content of the file you specify by `%%hey answer_file=foo.s`


# 2. Observe: compile example functions
* Before writing your own assembly, it helps to see what the compiler generates for related example functions.
* Running the first cell below writes `explore.c` (some small example functions related to this problem).
* The second cell compiles it with `gcc -O3 -S` and prints the generated assembly.
* Feel free to edit `explore.c` (change the code, add functions, change constants) and re-run the two cells to see how the assembly changes.

In [ ]:
%%writefile_ explore.c
/* When a function calls another function, it must set up a stack frame
   (save x29/x30) before the call. Observe the stp/ldp and bl instructions. */
#include <math.h>
double sigmoid(double x) { return 1.0 / (1.0 + exp(-x)); }

In [ ]:
%%bash_
gcc -O3 -S explore.c
cat explore.s


# 3. Your Answer (assembly)
* Running the cell below writes the skeleton assembly file `expsum.s`.
* Fill in your instructions after the line `// ------- write your answer here -------`, then run the cell again to save it.

In [ ]:
%%writefile_ expsum.s
	.arch armv8-a
	.file	"expsum.c"
	.text
	.align	2
	.p2align 4,,11
	.global	expsum
	.type	expsum, %function
expsum:
.LFB0:
	.cfi_startproc
	// ------- write your answer here -------
	.cfi_endproc
.LFE0:
	.size	expsum, .-expsum
	.ident	"GCC: (Ubuntu 13.3.0-6ubuntu2~24.04) 13.3.0"
	.section	.note.GNU-stack,"",@progbits


# 4. Checker
* The following C program calls your `expsum` function and checks the result against a reference computed in C.

In [ ]:
%%writefile_ check_expsum.c
#include <assert.h>
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
double expsum(double x);

int main(int argc, char ** argv) {
  assert(argc == 2);
  double x = atof(argv[1]);
  double r = expsum(x);
  double rc = exp(x) + exp(-x);
  if (fabs(r - rc) <= 1e-9 * (1.0 + fabs(rc))) {
    printf("OK %f %f\n", r, rc);
    return 0;
  } else {
    printf("NG %f %f\n", r, rc);
    return 1;
  }
}


# 5. Compile
* Compile your assembly together with the checker.
* If you get an error, fix `expsum.s` above and recompile.

In [ ]:
%%bash_
gcc -o check_expsum -O3 check_expsum.c expsum.s -lm


# 6. Run
* The commands to run the checker are stored in `run.sh`.
* If you see `OK`s and no errors, you are done.

In [ ]:
%%bash_
./check_expsum 0.0
./check_expsum 1.0
./check_expsum 2.0
./check_expsum -1.5


# 7. If things do not go well
* If your program compiles but does not produce the correct answer, run it within a debugger (gdb).
* Compile with `-O0 -g` first:
```
gcc -o check_expsum -O0 -g check_expsum.c expsum.s -lm
```
* Then, in a terminal (SSH or Jupyter terminal):
```
gdb check_expsum
(gdb) break expsum
(gdb) run ...        # give the same arguments as in run.sh
```
* Step through one instruction at a time with `step`, and inspect registers with `print $x0` or `info registers`.

# 8. Ask Questions or Get Feedback
* You are encouraged to ask for feedback once you think you are done, to know if there is a better answer.

In [ ]:
%%hey problem_file=expsum.md answer_file=expsum.s

Problem:
{problem}

My Answer:
{answer}

Give me a feedback to my answer.